# IOAI — 2024 First Stage Dependency Parsing (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
if not os.path.exists('data/train.conll'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-first-stage-dependency-parsing/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 의존 구문 분석 — 구조 프로브 (Dependency Parsing)

폴란드 AI 올림피아드 2024 (1차 예선). 폴란드어 문장의 **의존 트리**(각 어절의 지배소=head)를
복원한다. 핵심 아이디어는 **Hewitt-Manning 구조 프로브**: 사전학습 언어모델(**HerBERT**)의 어절
임베딩에 선형 변환 B 를 학습해, 두 어절 사이의 **트리 거리**와 각 어절의 **깊이**(루트로부터의 거리)를
회귀한다. 예측 거리로 **최소신장트리(MST)** 를 만들어 무방향 의존 트리를, 예측 깊이로 **루트**를 고른다.

- **데이터**: `data/train.conll`(1000문장)·`data/valid.conll`(200문장). CoNLL 형식(2열=어절, 7열=head, 0=루트).
- **제출**: `submission.csv` — `sent_id,token_id,head` (token_id 1-기반, head 1-기반, 0=루트).
- **채점**: **UUAS**(무방향 어절부착 정확도) + **root placement**(루트 정확도) → 원본 배점
  `points = scale(root) + scale(uuas)`, 각 `scale(x)= (clip(x,0.5,0.85)-0.5)/0.35`, 총 **0~2**.

이 노트북은 **베이스라인**(추적/학습 없이 선형 체인) 이다. 모범답안(구조 프로브)을 참고하라.


In [ ]:
# 데이터 준비 (Colab: 자동 다운로드 / DGX: data/ 이미 존재)
import os, urllib.request, zipfile
if not os.path.exists("data/train.conll"):
    url = "https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-first-stage-dependency-parsing/data.zip"
    urllib.request.urlretrieve(url, "d.zip"); zipfile.ZipFile("d.zip").extractall("data")
print("데이터:", sorted(os.listdir("data")))


In [ ]:
import csv, collections

def read_conll(fp):
    """CoNLL: 각 줄 '<idx>\t<word>\t...\t<head>\t...'; 빈 줄이 문장 경계. head 0 = 루트."""
    sents, words, heads = [], [], []
    for line in open(fp, encoding="utf-8"):
        sp = line.strip().split("\t")
        if len(sp) >= 7:
            words.append(sp[1]); heads.append(int(sp[6]))
        elif words:
            sents.append((words, heads)); words, heads = [], []
    if words: sents.append((words, heads))
    return sents

valid = read_conll("data/valid.conll")
print("valid 문장:", len(valid), "| 예:", valid[0][0])


In [ ]:
def parse_chain(words):
    """베이스라인: 각 어절을 바로 앞 어절에 붙이는 선형 체인(첫 어절=루트).
    (인접 어절이 실제로 연결된 경우가 많아 UUAS 는 0.5 부근이지만 트리 구조는 못 잡는다.)"""
    n = len(words)
    return [0] + list(range(1, n))   # token0 head=0(root); token i head=i (앞 어절)

rows = []
for sid, (words, _) in enumerate(valid):
    heads = parse_chain(words)
    for tid, h in enumerate(heads):
        rows.append([sid, tid + 1, h])
with open("submission.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["sent_id", "token_id", "head"]); w.writerows(rows)
print("submission.csv 저장:", len(rows), "행")


### 다음 단계
`parse_chain` 을 **구조 프로브**로 대체하라: HerBERT 임베딩 → 거리/깊이 선형 프로브 학습 →
MST 로 간선 선택, 예측 깊이로 루트 선택. 모범답안 노트북 참고.


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)